In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import classification_report, confusion_matrix
from transformers import pipeline

In [ ]:
#df_spam = pd.read_csv('/content/sample_data/spam_clean.csv', encoding='latin-1')

df_spam = pd.read_csv('../datas/spam_clean.csv', encoding='latin-1') 
# This is the one in use in this folder, but this whole notebook was executed on Google Collab using the one above

df_spam.head()

,label_string,message,label
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [9]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="roshana1s/spam-message-classifier")

Device set to use cpu


In [13]:
from transformers import AutoTokenizer

# Load model directly
checkpoint = "roshana1s/spam-message-classifier"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [14]:
def tokenize_function(example):
    return tokenizer(example["message"], truncation=True, padding=True)

In [15]:
from datasets import Dataset

# Conversion pandas -> Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_spam)

# Split 80% train, 20% test (ou val)
split_dataset = hf_dataset.train_test_split(test_size=0.2)

# Accès aux sous-datasets
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

# Appliquer la fonction de tokenization avec batching
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [16]:
from transformers import DataCollatorWithPadding

# Auto Padding : all sequences in a batch are padded to the same length
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [17]:
samples = tokenized_train_dataset[:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[122, 122, 122, 122, 122, 122, 122, 122]

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer", report_to="none")

In [19]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

In [20]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,0.085800
1000,0.023000
1500,0.016900


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1674, training_loss=0.037736085723093046, metrics={'train_runtime': 18418.1168, 'train_samples_per_second': 0.726, 'train_steps_per_second': 0.091, 'total_flos': 1297909363872360.0, 'train_loss': 0.037736085723093046, 'epoch': 3.0})

On Google Collab, it predicts the execution would take about 6 hours.


In [22]:
predictions = trainer.predict(tokenized_val_dataset)
print(predictions.predictions.shape, predictions.label_ids.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


(1115, 2) (1115,)


In [23]:
preds = np.argmax(predictions.predictions, axis=-1)
preds

array([0, 0, 0, ..., 0, 0, 0])

In [24]:
print(classification_report(tokenized_val_dataset["label"], preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       974
           1       1.00      0.98      0.99       141

    accuracy                           1.00      1115
   macro avg       1.00      0.99      0.99      1115
weighted avg       1.00      1.00      1.00      1115



In [25]:
# produce the confusion matrix for your predictions, what comments can you make ?
mat = confusion_matrix(tokenized_val_dataset["label"], preds)

labels = df_spam["label_string"].unique()
df_mat = pd.DataFrame(mat, index=labels, columns=labels)

fig = px.imshow(df_mat, text_auto=True, color_continuous_scale=px.colors.sequential.Agsunset,
    labels=dict(x="Prediction", y="Real", color="Nombre"))
fig.update_coloraxes(showscale=False)
fig.show()

# Conclusion

In the end, it took about 5 hours to get about the exact same result as with the home-made lighter and faster model.

So while it may be relevant to use a pretrained model on very large datasets, here in our case, with a relatively small sample, few ressources and a fairly well optimized classification method, it is hard to justify using a lot of time and a lot of computing power for the same result.